In [3]:
import pandas as pd
from bs4 import BeautifulSoup
import re
import datetime as date
import requests


URL = URL = "https://www.ccny.cuny.edu/registrar/fall"

In [4]:
response = requests.get(URL, headers = {"user": "Chrome (compatible; ccny-calendar-scraper)"}, timeout = 60, )
response.raise_for_status() #Creating a get request from the CCNY Website
html = response.text
response.status_code


200

In [5]:
soup = BeautifulSoup(html, "html.parser")

main = soup.find("main")
if main:
    text = main.get_text("\n", strip=True)
else:
    text = soup.get_text("\n", strip=True)

start_marker = "Fall 2021 Academic Calendar"
start_idx = text.find(start_marker)

if start_idx != -1:
    calendar_text = text[start_idx:]
else:
    calendar_text=text

In [ ]:
MONTHS = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

def infer_date(month):
    if month == "January":
        return 2022
    return 2021

def parse_events(lines):
    events = []
    i =0

    while i < len(lines):
        line = lines[i].strip()

        parts= line.split()
        if len(parts) >= 2 and parts[0] in MONTHS and parts[1].isdigit():
            month = parts[0]
            day = int(parts[1])
            year = infer_date(month)

            dow = ""
            if i +1 <len(lines):
                dow=lines[i+1].strip()

            text=""
            if i+2 <len(lines):
                text=lines[i+2].strip()

            dt=pd.Timestamp(year=year, month=MONTHS.index(month) + 1, day=day).date()

            events.append({"date": dt,
                           "dow": dow,
                           "text": text})
            i+=3
        else:
            i+=1
    return events

lines = calendar_text.splitlines()
events = parse_events(lines)

